# Lab 15-03: ACA Evaluation Job

This notebook:
1. Uploads evaluation data and the evaluation script to Blob Storage (via `DefaultAzureCredential`)
2. Submits a GPU evaluation job to Azure Container Apps
3. Downloads results and renders accuracy comparison chart

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
repo_root = Path.cwd().parents[0]
load_dotenv(repo_root / '.env', override=True)

In [ ]:
import os
import matplotlib.pyplot as plt
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
import importlib
import azure_infra
importlib.reload(azure_infra)
from azure_infra import upload_eval_data, submit_evaluation_job, monitor_job, download_eval_results

In [ ]:
# Configuration from environment variables
RESOURCE_GROUP = os.getenv("FINETUNE_RESOURCE_GROUP")
STORAGE_ACCOUNT = os.getenv("FINETUNE_STORAGE_ACCOUNT")
ACA_ENV = os.getenv("FINETUNE_ACA_ENVIRONMENT")
BASE_MODEL_ID = "microsoft/Phi-4-mini-instruct"
LOCATION = "swedencentral"
CONTAINER_NAME = "ft"
EVAL_JOB_NAME = "iss-eval-job"

# Reference values from 15-01 (re-load or use known values)
# These would be passed from 15-01 in a full pipeline; set fallbacks here
base_acc = 0.457

## 5. Evaluate Fine-Tuned Model on Azure Container Apps (Serverless GPU)

Run inference on Azure Container Apps with A100 GPU.

In [ ]:
# 1. Upload evaluation data to blob storage
print("Uploading evaluation data...")
upload_eval_data(STORAGE_ACCOUNT, CONTAINER_NAME, eval_dataset, reports_data, BASE_MODEL_ID)

# 2. Submit evaluation job to ACA
print("\nSubmitting evaluation job...")
submit_evaluation_job(
    EVAL_JOB_NAME, RESOURCE_GROUP, env_id,
    STORAGE_ACCOUNT, CONTAINER_NAME,
    BASE_MODEL_ID, LOCATION
)

# 3. Monitor (~5-10 minutes)
print("\nMonitoring evaluation job...")
eval_success = monitor_job(EVAL_JOB_NAME, RESOURCE_GROUP)

if eval_success:
    print("\n✅ Evaluation complete!")
else:
    print("\n❌ Evaluation failed. Check Azure Portal for logs.")

In [ ]:
# Download and display evaluation results
if eval_success:
    eval_results = download_eval_results(STORAGE_ACCOUNT, CONTAINER_NAME)
    ft_acc = eval_results["accuracy"]
    
    print(f"✨ Fine-Tuned Accuracy: {ft_acc:.1%}")
    print(f"\nResults breakdown:")
    
    # Show mismatches
    mismatches = [r for r in eval_results["results"] if not r["exact_match"]]
    if mismatches:
        print(f"\nMismatches ({len(mismatches)}):")
        for m in mismatches[:5]:
            print(f"  {m['date']}: Expected {m['expected']}, Got {m['predicted']}")
else:
    ft_acc = 0.0
    print("Using fallback accuracy of 0% due to job failure")

## 6. Performance Comparison

Compare Teacher, Baseline, and Fine-Tuned model accuracies.

In [ ]:
# Performance Comparison
models = ['gpt-4.1-mini\n(Teacher)', 'Phi-4-mini\n(Base)', 'Phi-4-mini\n(Fine-Tuned)']
accuracies = [accuracy, base_acc, ft_acc]
colors = ['#4CAF50', '#9E9E9E', '#2196F3']

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=colors, edgecolor='black', linewidth=1.2)

# Add value labels
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
             f'{acc:.1%}', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.title('ISS Incident Classification Accuracy\n(Teacher: gpt-4.1-mini via APIM)', fontsize=16, fontweight='bold')
plt.ylabel('Accuracy', fontsize=12)
plt.ylim(0, 1.1)
plt.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Target (80%)')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Summary Table
training_count = len(all_training) if 'all_training' in dir() else "N/A"
comparison_df = pd.DataFrame({
    "Model": ['DeepSeek-V3.2 (Teacher)', 'Phi-4-mini (Base)', 'Phi-4-mini (Fine-Tuned)'],
    "Accuracy": [f"{acc:.1%}" for acc in accuracies],
    "Training Data": ["N/A", "N/A", f"{training_count} examples"],
    "Inference": ["Cloud API", "Reference", "ACA GPU (A100)"]
})
display(comparison_df)

# Improvement calculation
improvement = ft_acc - base_acc
print(f"\n📈 Improvement from fine-tuning: {improvement:+.1%}")